In [24]:



import pandas as pd

In [ ]:
chartevents_original_df = pd.read_pickle('/Users/riccardoconci/Local_documents/Counterfactual_ICU/data/mimic_3_data/temp_dfs/CHARTEVENTS_chartevents.pkl')
chartevents_original_df = chartevents_original_df[chartevents_original_df["value"] > 0]
chartevents_original_df.rename(columns={'itemid': 'item_id'}, inplace=True)
chartevents_original_df

,subject_id,hadm_id,item_id,charttime,value,valueuom
1,124,138376,224690,2166-01-16 12:00:00,23.00,insp/min
2,124,138376,224695,2166-01-16 12:00:00,23.00,cmH2O
3,124,138376,224697,2166-01-16 12:00:00,12.00,cmH2O
4,124,138376,224738,2166-01-16 12:00:00,0.85,sec
5,124,138376,225309,2166-01-16 12:00:00,126.00,mmHg
...,...,...,...,...,...,...
18980537,99777,197851,220739,2167-09-16 08:00:00,4.00,NaN
18980538,99777,197851,223791,2167-09-16 08:00:00,4.00,NaN
18980539,99777,197851,223849,2167-09-16 08:00:00,30.00,NaN
18980540,99777,197851,223900,2167-09-16 08:00:00,5.00,NaN


In [27]:
items_df = pd.read_csv('/Users/riccardoconci/Local_documents/Counterfactual_ICU/data/mimic_3_data/input_data/D_ITEMS.csv')
items_df.rename(columns={'ITEMID': 'itemid', 'LABEL': 'label'}, inplace=True)


In [28]:
def attach_item_labels(inputs_df: pd.DataFrame, items_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a string label for each item_id using D_ITEMS.
    Creates a new column 'item_label'. If no match, falls back to str(item_id).
    """
    # tolerate case differences (MIMIC-III/IV)
    items = items_df.rename(columns=str.lower)
    if not {"itemid", "label"}.issubset(items.columns):
        raise ValueError("items_df must have columns ITEMID and LABEL (any case).")

    items_slim = items[["itemid", "label"]].drop_duplicates()

    df = inputs_df.merge(items_slim, left_on="item_id", right_on="itemid", how="left")
    df = df.drop(columns=["itemid"])
    df["item_label"] = df["label"].fillna(df["item_id"].astype(str))
    df = df.drop(columns=["label"])  # keep things tidy (label now lives in item_label)
    return df

In [ ]:
chartevents_original_df = attach_item_labels(chartevents_original_df, items_df)


In [34]:
chartevents_original_df.head()

,subject_id,hadm_id,item_id,charttime,value,valueuom,item_label
0,124,138376,224690,2166-01-16 12:00:00,23.00,insp/min,Respiratory Rate (Total)
1,124,138376,224695,2166-01-16 12:00:00,23.00,cmH2O,Peak Insp. Pressure
2,124,138376,224697,2166-01-16 12:00:00,12.00,cmH2O,Mean Airway Pressure
3,124,138376,224738,2166-01-16 12:00:00,0.85,sec,Inspiratory Time
4,124,138376,225309,2166-01-16 12:00:00,126.00,mmHg,ART BP Systolic


In [ ]:
top_items = chartevents_original_df['item_id'].value_counts()[:100]

In [35]:
chartevents_original_df_top_items = chartevents_original_df[chartevents_original_df['item_id'].isin(top_items.index)]

In [36]:
chartevents_original_df_top_items

,subject_id,hadm_id,item_id,charttime,value,valueuom,item_label
0,124,138376,224690,2166-01-16 12:00:00,23.0,insp/min,Respiratory Rate (Total)
1,124,138376,224695,2166-01-16 12:00:00,23.0,cmH2O,Peak Insp. Pressure
2,124,138376,224697,2166-01-16 12:00:00,12.0,cmH2O,Mean Airway Pressure
6,124,138376,225312,2166-01-16 12:00:00,66.0,mmHg,ART BP mean
9,124,138376,220045,2166-01-16 13:00:00,75.0,bpm,Heart Rate
...,...,...,...,...,...,...,...
18366388,99776,136231,223900,2171-07-16 12:00:00,4.0,NaN,GCS - Verbal Response
18366389,99776,136231,223901,2171-07-16 12:00:00,6.0,NaN,GCS - Motor Response
18366404,99777,197851,220739,2167-09-16 08:00:00,4.0,NaN,GCS - Eye Opening
18366407,99777,197851,223900,2167-09-16 08:00:00,5.0,NaN,GCS - Verbal Response


In [45]:
dict(chartevents_original_df[['item_label', 'item_id']].value_counts()[:100])

{('Heart Rate', 211): np.int64(673369),
 ('calprevflg', 742): np.int64(669691),
 ('SpO2', 646): np.int64(666270),
 ('Respiratory Rate', 618): np.int64(650749),
 ('Arterial BP [Systolic]', 51): np.int64(479226),
 ('Arterial BP [Diastolic]', 8368): np.int64(479130),
 ('Arterial BP Mean', 52): np.int64(476254),
 ('Previous WeightF', 581): np.int64(322486),
 ('Heart Rate', 220045): np.int64(251732),
 ('CVP', 113): np.int64(249654),
 ('Respiratory Rate', 220210): np.int64(248552),
 ('O2 saturation pulseoxymetry', 220277): np.int64(243258),
 ('HR Alarm [High]', 8549): np.int64(235953),
 ('HR Alarm [Low]', 5815): np.int64(235915),
 ('SpO2 Alarm [Low]', 5820): np.int64(235608),
 ('SpO2 Alarm [High]', 8554): np.int64(234814),
 ('NBP [Systolic]', 455): np.int64(233175),
 ('NBP [Diastolic]', 8441): np.int64(233084),
 ('Resp Alarm [High]', 8553): np.int64(232714),
 ('Resp Alarm [Low]', 5819): np.int64(232497),
 ('NBP Mean', 456): np.int64(230723),
 ('Eye Opening', 184): np.int64(170889),
 ('Verbal

In [57]:
def normalize_value_with_ranges(df, value_col='value', group_col='item_label'):
    """
    Normalizes the values for each itemid in the DataFrame based on their own mean and std.
    Any standardized values greater than 3 or less than -3 (i.e. outliers) are clipped to ±3.

    Additionally, returns the corresponding original ranges of ±3 std for each group, so that
    normalized values can be mapped back to the original scale.

    Parameters:
        df (pd.DataFrame): The input DataFrame containing the lab data.
        value_col (str): The name of the column containing the values to normalize.
        group_col (str): The column name by which to group the data (each unique itemid).

    Returns:
        df_normalized (pd.DataFrame): A new DataFrame with the normalized and clipped values.
        df_ranges (pd.DataFrame): A DataFrame with original ±3 std ranges for each group.
    """
    # Compute the mean and std for each group (each itemid)
    means = df.groupby(group_col)[value_col].transform('mean')
    stds = df.groupby(group_col)[value_col].transform('std')
    
    # Standardize the values: (value - mean) / std
    normalized = (df[value_col] - means) / stds
    
    # Clip the standardized values to the range [-3, 3]
    normalized_clipped = normalized.clip(lower=-3, upper=3)
    
    # Create a copy of the DataFrame with the normalized values
    df_normalized = df.copy()
    df_normalized[value_col + "_normalized"] = normalized_clipped
    
    # Create a DataFrame with original ranges per group
    group_stats = df.groupby(group_col)[value_col].agg(['mean', 'std']).reset_index()
    group_stats['lower_bound'] = group_stats['mean'] - 3 * group_stats['std']
    group_stats['upper_bound'] = group_stats['mean'] + 3 * group_stats['std']
    
    # Add counts for sorting
    counts = df[group_col].value_counts().reset_index()
    counts.columns = [group_col, 'count']
    
    df_ranges = group_stats[[group_col, 'lower_bound', 'upper_bound']].merge(
        counts, on=group_col, how='left'
    ).sort_values(by='count', ascending=False).reset_index(drop=True)
    
    return df_normalized, df_ranges



In [58]:
chartevents_original_df_top_items_normalised, chartevents_original_df_top_items_ranges = normalize_value_with_ranges(chartevents_original_df_top_items)

In [59]:
chartevents_original_df_top_items_normalised

,subject_id,hadm_id,item_id,charttime,value,valueuom,item_label,value_normalized
0,124,138376,224690,2166-01-16 12:00:00,23.0,insp/min,Respiratory Rate (Total),0.083316
1,124,138376,224695,2166-01-16 12:00:00,23.0,cmH2O,Peak Insp. Pressure,-0.048602
2,124,138376,224697,2166-01-16 12:00:00,12.0,cmH2O,Mean Airway Pressure,0.218133
6,124,138376,225312,2166-01-16 12:00:00,66.0,mmHg,ART BP mean,-0.175291
9,124,138376,220045,2166-01-16 13:00:00,75.0,bpm,Heart Rate,-0.735385
...,...,...,...,...,...,...,...,...
18366388,99776,136231,223900,2171-07-16 12:00:00,4.0,NaN,GCS - Verbal Response,0.860879
18366389,99776,136231,223901,2171-07-16 12:00:00,6.0,NaN,GCS - Motor Response,0.610516
18366404,99777,197851,220739,2167-09-16 08:00:00,4.0,NaN,GCS - Eye Opening,0.781109
18366407,99777,197851,223900,2167-09-16 08:00:00,5.0,NaN,GCS - Verbal Response,1.411833


In [62]:
chartevents_original_df_top_items_ranges.head(40)

,item_label,lower_bound,upper_bound,count
0,Heart Rate,33.875181,142.833605,925101
1,Respiratory Rate,-1.450279,43.054465,899301
2,calprevflg,1.000000,1.000000,669691
3,SpO2,86.950721,107.620443,666270
4,Arterial BP [Systolic],46.445002,197.835606,479226
5,Arterial BP [Diastolic],18.393587,100.994078,479130
6,Arterial BP Mean,26.718321,134.479583,476254
7,Previous WeightF,12.057408,161.132925,322486
8,CVP,-5.157119,28.737937,249654
9,O2 saturation pulseoxymetry,-4.604344,199.796067,243258


In [90]:
# Convert [T, K] to [B, T, K] with B=4 by repeating along new batch dimension
mock_batch_chartevent = chartevents_sample[0].unsqueeze(0).repeat(4, 1, 1)

In [92]:
mock_batch_chartevent.shape

torch.Size([4, 24, 192])

In [119]:
import torch
import pandas as pd

import torch
import pandas as pd

def _normalize_label(s: str) -> str:
    """Normalize label text for matching across index_map and df_ranges."""
    return str(s).strip().lower().replace("_", " ").replace("-", " ")

def _stats_tensors_from_ranges(
    df_ranges: pd.DataFrame,
    labels,
    group_col: str = 'item_label',
    device=None,
    dtype=torch.float32
):
    """
    Build per-feature mean/std/lower/upper tensors (shape [K]) from df_ranges.
    Label matching is normalized (lowercased, underscores -> spaces, etc.).
    """
    required = {'lower_bound', 'upper_bound', group_col}
    if not required.issubset(df_ranges.columns):
        missing = required - set(df_ranges.columns)
        raise ValueError(f"df_ranges is missing required columns: {missing}")

    # Normalize df_ranges labels
    df_ranges = df_ranges.copy()
    df_ranges['_norm_label'] = df_ranges[group_col].map(_normalize_label)
    stats = df_ranges.set_index('_norm_label')[['lower_bound', 'upper_bound']]

    # Normalize the labels list
    norm_labels = [_normalize_label(l) for l in labels]

    # Build tensors in the order of labels
    try:
        lb = torch.tensor([float(stats.loc[l, 'lower_bound']) for l in norm_labels],
                          device=device, dtype=dtype)
        ub = torch.tensor([float(stats.loc[l, 'upper_bound']) for l in norm_labels],
                          device=device, dtype=dtype)
    except KeyError as e:
        raise KeyError(f"Label '{e.args[0]}' not found after normalization in df_ranges[{group_col}].")

    mean = (lb + ub) / 2.0
    std  = (ub - lb) / 6.0
    return mean, std, lb, ub

def denormalize_selected_from_batched_sample(
    chartevents_sample: torch.Tensor,
    df_ranges: pd.DataFrame,
    index_map: dict,
    group_col: str = 'item_label',
    d_inp: int = 96,
    use_mask: bool = True,
    clip: bool = True,
):
    """
    Vectorized inverse-normalization for a batched tensor.

    Args:
      chartevents_sample: torch.Tensor of shape [B, T, 2*d_inp]
                          first d_inp = z-scored values, next d_inp = presence mask (0/1 or bool)
      df_ranges: DataFrame with columns [group_col, lower_bound, upper_bound] (bounds are mean ± 3*std)
      index_map: dict mapping item_label -> feature index (column in X) for features of interest
      group_col: column name in df_ranges that matches item labels in index_map
      d_inp: number of clinical features (e.g., 96)
      use_mask: if True use the provided presence mask; otherwise treat non-NaN as present
      clip: clip z to [-3, 3] before inverse-transform (recommended)

    Returns (all torch tensors):
      original_values: [B, K] inverse-mapped to original scale
      z_values:        [B, K] raw z-scores extracted at the last present timestep
      timesteps:       [B, K] int64 indices of last present timestep (=-1 where none present)
      means:           [K]
      stds:            [K]
      lower_bounds:    [K]
      upper_bounds:    [K]
      present_mask:    [B, K] bool, True where a present value was found
    """
    if not torch.is_tensor(chartevents_sample) or chartevents_sample.dim() != 3:
        raise ValueError("chartevents_sample must be a torch.Tensor of shape [B, T, 2*d_inp].")

    B, T, F_total = chartevents_sample.shape
    if F_total != 2 * d_inp:
        raise ValueError(f"Expected last dim = 2*d_inp ({2*d_inp}), got {F_total}.")

    device = chartevents_sample.device
    dtype = chartevents_sample.dtype

    # Split into values and mask
    X = chartevents_sample[:, :, :d_inp]           # [B, T, d_inp]
    M = chartevents_sample[:, :, d_inp:]           # [B, T, d_inp]

    # Indices/labels of interest
    labels = list(index_map.keys())
    feat_idx = torch.tensor([index_map[l] for l in labels], device=device, dtype=torch.long)
    if (feat_idx < 0).any() or (feat_idx >= d_inp).any():
        raise IndexError("One or more feature indices in index_map are out of bounds for d_inp.")

    # Select features of interest
    X_sel = X.index_select(dim=2, index=feat_idx)                # [B, T, K]
    if use_mask:
        # Convert to boolean presence (accepts float 0/1, int, or bool)
        M_sel = M.index_select(dim=2, index=feat_idx)
        if M_sel.dtype != torch.bool:
            M_sel = M_sel > 0.5
    else:
        M_sel = ~torch.isnan(X_sel)

    # Compute last present timestep per (B,K)
    time_idx = torch.arange(T, device=device).view(1, T, 1)      # [1, T, 1]
    # Put 0 where absent, then take max; fix up later where none are present
    idx_weighted = torch.where(M_sel, time_idx, torch.zeros(1, dtype=time_idx.dtype, device=device))
    last_idx = idx_weighted.amax(dim=1)                          # [B, K], 0 if none present
    any_present = M_sel.any(dim=1)                               # [B, K] bool
    last_idx = torch.where(any_present, last_idx, torch.full_like(last_idx, -1))

    # Gather z-values at the last present timestep
    last_idx_safe = torch.clamp(last_idx, min=0)                 # [B, K]
    z_values = X_sel.gather(1, last_idx_safe.unsqueeze(1).expand(-1, 1, -1)).squeeze(1)  # [B, K]
    z_values = torch.where(any_present, z_values, torch.full_like(z_values, float('nan')))

    # Stats in K-order
    means, stds, lower_bounds, upper_bounds = _stats_tensors_from_ranges(
        df_ranges, labels, group_col=group_col, device=device, dtype=torch.float32
    )
    means = means.to(dtype)
    stds = stds.to(dtype)
    lower_bounds = lower_bounds.to(dtype)
    upper_bounds = upper_bounds.to(dtype)

    # Inverse transform
    z_use = torch.clamp(z_values, -3.0, 3.0) if clip else z_values
    original_values = z_use * stds.view(1, -1) + means.view(1, -1)              # [B, K]
    original_values = torch.where(any_present, original_values, torch.full_like(original_values, float('nan')))

    return (
        original_values,         # [B, K]
        z_values,                # [B, K]
        last_idx.to(torch.long), # [B, K]
        means,                   # [K]
        stds,                    # [K]
        lower_bounds,            # [K]
        upper_bounds,            # [K]
        any_present              # [B, K] bool
    )

In [120]:
chartevents_original_df_top_items_ranges.iloc[0]

item_label     Heart Rate
lower_bound     33.875181
upper_bound    142.833605
count              925101
Name: 0, dtype: object

In [121]:
indx_interest = {'Heart rate': 0, 'Arterial BP Mean': 6, 'CVP': 8}
out = denormalize_selected_from_batched_sample(mock_batch_chartevent, chartevents_original_df_top_items_ranges, indx_interest, group_col='item_label')
original_values, z_values, timesteps, means, stds, lb, ub, present = out

In [122]:
original_values

tensor([[88.3544, 80.5990, 11.7904],
        [88.3544, 80.5990, 11.7904],
        [88.3544, 80.5990, 11.7904],
        [88.3544, 80.5990, 11.7904]])

In [ ]:

import torch

chartevents_sample = torch.load('/Users/riccardoconci/Local_documents/Counterfactual_ICU/data/mimic_3_data/processed_data/processed_data_3/chartevents_tensors_output/chartevents_context/chartevents_context_100098_53.pt', weights_only=False)

num_ids = 96
t_time = 24
chartevents_sample[0][-1, :num_ids]



In [74]:
original_values

tensor([[ 85.9017, 101.7292,   8.6146],
        [ 88.7236, 138.2007,   9.4762]])

In [ ]:
chartevents_df = pd.read_parquet('/Users/riccardoconci/Local_documents/Counterfactual_ICU/data/mimic_3_data/CHARTEVENTS_top_items_normalized_selected.parquet')
chartevents_df.head()



,subject_id,hadm_id,item_id,charttime,value,valueuom,item_label
0,124,138376,224690,2166-01-16 12:00:00,0.083316,insp/min,Respiratory Rate (Total)
1,124,138376,224695,2166-01-16 12:00:00,0.488865,cmH2O,Peak Insp. Pressure
2,124,138376,224697,2166-01-16 12:00:00,0.425271,cmH2O,Mean Airway Pressure
6,124,138376,225312,2166-01-16 12:00:00,-0.175291,mmHg,ART BP mean
9,124,138376,220045,2166-01-16 13:00:00,-0.876172,bpm,Heart Rate


tensor([ 0.5142, -0.7099,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000, -0.0762,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.3705, -0.7006,
        -0.2588,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000, -0.2676,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.7811,  0.0259, -0.7920,  0.0000])

In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import math
from dataclasses import dataclass
from typing import Dict, Tuple, Optional


@dataclass
class PhysioRanges:
    # Default ranges (your values)
    p_a: Tuple[float, float] = (40.0, 180.0)
    p_v: Tuple[float, float] = (0.0, 30.0)
    s_reflex: Tuple[float, float] = (0.0, 1.0)
    sv: Tuple[float, float] = (40.0, 120.0)
    r_tpr_mod: Tuple[float, float] = (-1.0, 1.0)
    f_hr_max: Tuple[float, float] = (2.0, 3.0)
    f_hr_min: Tuple[float, float] = (0.9, 1.1)
    r_tpr_max: Tuple[float, float] = (1.8, 2.4)
    r_tpr_min: Tuple[float, float] = (0.45, 0.6)
    ca: Tuple[float, float] = (2.0, 6.0)
    cv: Tuple[float, float] = (90.0, 120.0)
    k_width: Tuple[float, float] = (0.1, 0.3)
    p_aset: Tuple[float, float] = (50.0, 90.0)
    tau: Tuple[float, float] = (15.0, 25.0)


def midpoint(r: Tuple[float, float]) -> float:
    return 0.5 * (r[0] + r[1])


def clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))


def compute_stable_equilibrium(
    Pa: float,
    Pv: float,
    Hr: float,
    *,
    # Optional parameters (if None, use midpoints from ranges)
    f_hr_min: Optional[float] = None,
    f_hr_max: Optional[float] = None,
    r_tpr_min: Optional[float] = None,
    r_tpr_max: Optional[float] = None,
    k_width: Optional[float] = None,
    # Strategy prefers r_tpr_mod = 0, only adjusts it if SV hits bounds
    r_tpr_mod_fixed: Optional[float] = 0.0,
    ranges: PhysioRanges = PhysioRanges(),
) -> Dict[str, float]:
    """
    Given Pa, Pv, Hr:
      1) Choose p_aset* to enforce baroreflex consistency (s_HR == s_baro) for stability.
      2) Set r_tpr_mod = 0 and compute SV* by flow balance.
      3) If SV* violates bounds, clip SV and solve for r_tpr_mod* instead.
      4) Report feasibility vs bounds.

    Returns dict with Pa, Pv, Hr, s, p_aset*, SV, R_tpr, r_tpr_mod, flags, etc.
    """

    # Fill defaults from midpoints if not provided
    if f_hr_min is None:
        f_hr_min = midpoint(ranges.f_hr_min)
    if f_hr_max is None:
        f_hr_max = midpoint(ranges.f_hr_max)
    if r_tpr_min is None:
        r_tpr_min = midpoint(ranges.r_tpr_min)
    if r_tpr_max is None:
        r_tpr_max = midpoint(ranges.r_tpr_max)
    if k_width is None:
        k_width = midpoint(ranges.k_width)

    if not (Pa > Pv):
        raise ValueError("Require Pa > Pv for positive arterial outflow.")
    if not (f_hr_max > f_hr_min):
        raise ValueError("Require f_hr_max > f_hr_min.")
    if not (Hr > 0.0):
        raise ValueError("Require Hr > 0.")

    # 1) Reflex consistency: s set by HR mapping; then solve p_aset* so ds/dt=0 holds at Pa
    s_HR = (Hr - f_hr_min) / (f_hr_max - f_hr_min)  # must lie in (0,1) for a valid p_aset*
    if not (0.0 < s_HR < 1.0):
        # Outside reflex range — cannot make HR consistent by adjusting p_aset
        # We'll still continue but mark infeasible_reflex=True
        infeasible_reflex = True
        # Clamp s for downstream arithmetic to avoid log blow-ups
        eps = 1e-6
        s_for_calc = clamp(s_HR, eps, 1.0 - eps)
    else:
        infeasible_reflex = False
        s_for_calc = s_HR

    # s_baro = 1 / (1 + exp(k*(Pa - p_aset)))  => solve for p_aset*
    # p_aset* = Pa - (1/k) * ln((1 - s)/s)
    p_aset_star = Pa - (1.0 / k_width) * math.log((1.0 - s_for_calc) / s_for_calc)

    # 2) Baseline resistance from reflex state s (no modulation yet)
    delta_r = (r_tpr_max - r_tpr_min)
    R_base = r_tpr_min + s_for_calc * delta_r

    # 3) Preferred: keep r_tpr_mod at provided fixed value (default 0) and compute SV*
    r_tpr_mod = 0.0 if r_tpr_mod_fixed is None else r_tpr_mod_fixed
    R_tpr = R_base + r_tpr_mod

    if R_tpr <= 0.0:
        raise ValueError("Computed total resistance R_tpr <= 0; adjust parameters or r_tpr_mod_fixed.")

    SV_star = (Pa - Pv) / (Hr * R_tpr)

    # 4) Enforce SV bounds; if violated, adjust r_tpr_mod instead
    SV_lo, SV_hi = ranges.sv
    rmod_lo, rmod_hi = ranges.r_tpr_mod

    if SV_lo <= SV_star <= SV_hi:
        # All good with r_tpr_mod as given (default 0)
        SV_final = SV_star
        r_tpr_mod_final = r_tpr_mod
        R_tpr_final = R_tpr
        used_clip = False
    else:
        # Clip SV to bounds and solve for r_tpr_mod to satisfy flow
        SV_clipped = clamp(SV_star, SV_lo, SV_hi)
        R_tpr_needed = (Pa - Pv) / (Hr * SV_clipped)
        r_tpr_mod_needed = R_tpr_needed - R_base

        # Enforce r_tpr_mod bounds
        r_tpr_mod_final = clamp(r_tpr_mod_needed, rmod_lo, rmod_hi)
        R_tpr_final = R_base + r_tpr_mod_final
        SV_final = (Pa - Pv) / (Hr * R_tpr_final)
        used_clip = True

    # Final s equals s_HR; check consistency (should be exact unless s_HR out of (0,1))
    # s_baro at p_aset*:
    s_baro = 1.0 / (1.0 + math.exp(k_width * (Pa - p_aset_star)))
    consistency_error = s_HR - s_baro  # want ~0

    # Feasibility flags
    within_sv_bounds = (SV_lo <= SV_final <= SV_hi)
    within_rmod_bounds = (rmod_lo <= r_tpr_mod_final <= rmod_hi)
    feasible = (not infeasible_reflex) and within_sv_bounds and within_rmod_bounds and (R_tpr_final > 0.0)

    return {
        # Inputs
        "Pa": Pa,
        "Pv": Pv,
        "Hr": Hr,
        # Reflex & setpoint
        "s": s_HR,
        "p_aset_star": p_aset_star,
        "s_baro_at_p_aset_star": s_baro,
        "consistency_error": consistency_error,
        "infeasible_reflex": float(infeasible_reflex),
        # Resistances and flows
        "R_base": R_base,
        "R_tpr": R_tpr_final,
        "SV": SV_final,
        "r_tpr_mod": r_tpr_mod_final,
        # Bounds & status
        "sv_bounds": ranges.sv,
        "r_tpr_mod_bounds": ranges.r_tpr_mod,
        "used_clip_or_rmod_adjust": float(used_clip),
        "within_sv_bounds": float(within_sv_bounds),
        "within_rmod_bounds": float(within_rmod_bounds),
        "feasible": float(feasible),
    }



In [15]:
Pa, Pv, Hr = 80, 1.0, 2  # mmHg, mmHg, Hz (~72 bpm)
res = compute_stable_equilibrium(Pa, Pv, Hr)
for k, v in res.items():
    print(f"{k}: {v}")

Pa: 80
Pv: 1.0
Hr: 2
s: 0.6666666666666666
p_aset_star: 83.46573590279972
s_baro_at_p_aset_star: 0.6666666666666665
consistency_error: 1.1102230246251565e-16
infeasible_reflex: 0.0
R_base: 1.5750000000000002
R_tpr: 0.9875
SV: 40.0
r_tpr_mod: -0.5875000000000001
sv_bounds: (40.0, 120.0)
r_tpr_mod_bounds: (-1.0, 1.0)
used_clip_or_rmod_adjust: 1.0
within_sv_bounds: 1.0
within_rmod_bounds: 1.0
feasible: 1.0


In [16]:
Pa, Pv, Hr = 80, 10.0, 2  # mmHg, mmHg, Hz (~72 bpm)
res = compute_stable_equilibrium(Pa, Pv, Hr)
for k, v in res.items():
    print(f"{k}: {v}")

Pa: 80
Pv: 10.0
Hr: 2
s: 0.6666666666666666
p_aset_star: 83.46573590279972
s_baro_at_p_aset_star: 0.6666666666666665
consistency_error: 1.1102230246251565e-16
infeasible_reflex: 0.0
R_base: 1.5750000000000002
R_tpr: 0.875
SV: 40.0
r_tpr_mod: -0.7000000000000002
sv_bounds: (40.0, 120.0)
r_tpr_mod_bounds: (-1.0, 1.0)
used_clip_or_rmod_adjust: 1.0
within_sv_bounds: 1.0
within_rmod_bounds: 1.0
feasible: 1.0
